# Bayes Decisions & Model Evaluation

Analyze the performance of the **MVG classifier and its variants** (Tied Gaussian, Naive Bayes) for different applications using Bayes decisions, DCF, minimum DCF, and Bayes error plots.

The **five applications** are expressed as $(\pi_1, C_{fn}, C_{fp})$:
- $(0.5, 1, 1)$ — uniform prior and costs
- $(0.9, 1, 1)$ — most users are genuine
- $(0.1, 1, 1)$ — most users are impostors
- $(0.5, 1, 9)$ — strong security: high cost for accepting an impostor
- $(0.5, 9, 1)$ — ease of use: high cost for rejecting a legit user

In [ ]:
import numpy as np
import scipy.special as ss
import matplotlib.pyplot as plt
import pandas as pd

## Helper functions

All utility functions for Bayes decisions and DCF computation.

In [ ]:
def vcol(x):
    """
        Convert the data to Column Vector
    """
    return x.reshape((x.size, 1))

def vrow(x):
    """
        Convert the data to Row Vector
    """
    return x.reshape((1, x.size))

In [ ]:
def compute_confusion_matrix(predictedLabels, classLabels):

    """
    Assume that classes are labeled 0, 1, 2 ... (nClasses - 1)

        - nClasses: Number of Classes
        - M: Confusion Matrix

    Rows = Predictions
    Columns = True Classes
    """
    nClasses = classLabels.max() + 1
    M = np.zeros((nClasses, nClasses), dtype=np.int32)
    for i in range(classLabels.size):
        M[predictedLabels[i], classLabels[i]] += 1
    return M

`computer_optimal_Bayes_binary_llr`:

In a binary task (where hypotheses are True $\mathcal{H}_T=1$ and False $\mathcal{H}_F=0$), we have two specific costs: $C_{fn}$ (the cost of a false negative) and $C_{fp}$ (the cost of a false positive). 

To minimize your expected risk, you can consolidate your prior beliefs ($\pi_1$) and your costs into a single decision threshold $t$:

$$t = -\log\frac{\pi_1 C_{fn}}{(1-\pi_1)C_{fp}}$$

In [ ]:
def compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp):
    """
    llr: Log - Likelihood Ratio

    Optimal Bayes decision for binary task with LLR scores.
    Compares LLR against threshold t = -log(pi1*Cfn / (1-pi1)*Cfp).
    
    Predict class 1 if LLR > t, else class 0.
    """
    th = -np.log((prior * Cfn) / ((1 - prior) * Cfp))
    return np.int32(llr > th)

- $P_{fn}$ **(False Negative Rate)**:

    It takes the number of False Negatives ($M_{0,1}$, which is Predicted 0, True 1) and divides it by the total number of actual Class 1 samples ($M_{0,1} + M_{1,1}$)


- $P_{fp}$ **(False Positive Rate)**:

    It takes the number of False Positives ($M_{1,0}$, which is Predicted 1, True 0) and divides it by the total number of actual Class 0 samples ($M_{0,0} + M_{1,0}$). 


- **The Math:** 
    
    For a binary task, the expected risk ($\mathcal{B}$ or $DCF_u$) is simply the sum of your two possible errors, weighted by their respective costs and prior probabilities.
    
    $DCF_u = \pi_1 C_{fn} P_{fn} + (1-\pi_1) C_{fp} P_{fp}$

    $\mathcal{B}_{dummy} = \min(\pi_1 C_{fn}, (1-\pi_1) C_{fp})$

    $DCF_{normalized} = \frac{DCF_u}{\mathcal{B}_{dummy}}$

In [ ]:
def compute_empirical_Bayes_risk_binary(predictedLabels, classLabels, prior, Cfn, Cfp, normalize=True):
    """
    DCF: Detection Cost Function

    Computes the (normalized) DCF = actual DCF.

    DCF_u = pi1*Cfn*Pfn + (1-pi1)*Cfp*Pfp
    Normalized DCF = DCF_u / min(pi1*Cfn, (1-pi1)*Cfp)
    
    """
    M = compute_confusion_matrix(predictedLabels, classLabels)
    
    Pfn = M[0,1] / (M[0,1] + M[1,1])  # False Negative Rate
    Pfp = M[1,0] / (M[0,0] + M[1,0])  # False Positive Rate
    
    bayesError = prior * Cfn * Pfn + (1-prior) * Cfp * Pfp
    
    if normalize:
        return bayesError / np.minimum(prior * Cfn, (1-prior)*Cfp)
    
    return bayesError

`compute_Pfn_Pfp_allThresholds_fast`

Your model just gave you a bunch of raw scores (LLRs) for your test data. To really know how good your model is, you can't just test one single decision boundary. You need to know how the model behaves if you shift the boundary from "accept everyone" all the way to "reject everyone."

This function is a mass-evaluation machine. It systematically tests every single meaningful threshold possible for the dataset and calculates your error rates at each one.

**Returns**

- `thOut` (**Thresholds**): The specific decision boundary being tested at that moment.

- `PfnOut` (**False Negative Rate**): At this specific threshold, what percentage of actual Class 1s did the model accidentally call Class 0?

- `PfOut` (**False Positive Rate**): At this specific threshold, what percentage of actual Class 0s did the model accidentally call Class 1?

In [ ]:
def compute_Pfn_Pfp_allThresholds_fast(llr, classLabels):
    """
    Efficiently sweeps all thresholds (sorted LLR values) and returns
    the Pfn, Pfp arrays and the corresponding thresholds.
    Used for computing minDCF and ROC curves.
    """
    llrSorter = np.argsort(llr)
    llrSorted = llr[llrSorter]
    classLabelsSorted = classLabels[llrSorter]
    nTrue = (classLabelsSorted==1).sum()
    nFalse = (classLabelsSorted==0).sum()
    nFalseNegative, nFalsePositive = 0, nFalse
    Pfn, Pfp = [nFalseNegative/nTrue], [nFalsePositive/nFalse]
    for idx in range(len(llrSorted)):
        if classLabelsSorted[idx] == 1: nFalseNegative += 1
        if classLabelsSorted[idx] == 0: nFalsePositive -= 1
        Pfn.append(nFalseNegative/nTrue)
        Pfp.append(nFalsePositive/nFalse)
    llrSorted = np.concatenate([-np.array([np.inf]), llrSorted])
    PfnOut, PfpOut, thOut = [], [], []
    for idx in range(len(llrSorted)):
        if idx == len(llrSorted)-1 or llrSorted[idx+1] != llrSorted[idx]:
            PfnOut.append(Pfn[idx]); PfpOut.append(Pfp[idx]); thOut.append(llrSorted[idx])
    
    # PfnOut, PfpOut, thOut
    return np.array(PfnOut), np.array(PfpOut), np.array(thOut)

- `compute_Pfn_Pfp_allThresholds_fast` **is the engine:** 
    
    It does all the hard work. It sweeps the line across the sorted scores and spits out the raw error rates ($P_{fn}$ and $P_{fp}$) for every single possible threshold.

- `compute_minDCF_binary_fast` **is just the calculator:** 

    It takes those massive lists of error rates, plugs them into the cost formula (which creates a massive list of DCF penalties), and then just looks at that list and says, "Which one of these numbers is the smallest?"

In [ ]:
def compute_minDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp):
    """
    Minimum DCF: the best normalized DCF achievable by sweeping all possible thresholds.
    This is a lower bound — it tells you how good the model COULD be with perfect calibration.
    """
    Pfn, Pfp, _ = compute_Pfn_Pfp_allThresholds_fast(llr, classLabels)

    minDCF = (prior*Cfn*Pfn + (1-prior)*Cfp*Pfp) / np.minimum(prior*Cfn, (1-prior)*Cfp)
    
    return minDCF.min()

## Load and split the project data

In [ ]:
def load_project_data(path):
    
    """
        Loads Project Data
    """

    D, L = [], []
    with open(path) as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(',')]
            D.append([float(x) for x in parts[:-1]])
            L.append(int(parts[-1]))
    return np.array(D).T, np.array(L, dtype=np.int32)

def split_db(D, L, seed=0, ratio=2/3):

    """
        Splits the Data
        - Training Data = 2/3 Data
        - Validationi Data = 1/3 Data
    """
    nTrain = int(D.shape[1] * ratio)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    return (D[:, idx[:nTrain]], L[idx[:nTrain]]), (D[:, idx[nTrain:]], L[idx[nTrain:]])

D, L = load_project_data('../../../Project/trainData.txt')
(DTR, LTR), (DVAL, LVAL) = split_db(D, L)
print('Training set:', DTR.shape)
print('Validation set:', DVAL.shape)

## Gaussian model parameter estimation

Reusing the same MVG, Tied and Naive Bayes models from Lab 5.

In [ ]:
def compute_mu_C(D):
    """
    Compute Mean & Covariance Matrix
        - mu: Mean 
        - C: Covariance
    """
    mu = vcol(D.mean(1))
    DC = D - mu
    return mu, (DC @ DC.T) / D.shape[1]

def logpdf_GAU_ND(X, mu, C):
    """
    log-Probability Density Function(PDF), N-Dimensional

    Given a bell curve centered at a specific point (mu),
    and stretched in a specific way (C),
    how likely are we to find a sample exactly at point (X)?

    """
    XC = X - mu
    M = X.shape[0]
    _, logdet = np.linalg.slogdet(C)
    invC = np.linalg.inv(C)
    return -0.5*M*np.log(2*np.pi) - 0.5*logdet - 0.5*((XC)*(invC@XC)).sum(0)

def mvg_params(DTR, LTR):
    return {c: compute_mu_C(DTR[:, LTR==c]) for c in sorted(set(LTR))}

def tied_params(DTR, LTR):
    means, C_tied = {}, 0
    for c in sorted(set(LTR)):
        Dc = DTR[:, LTR==c]
        mu, Cc = compute_mu_C(Dc)
        means[c] = mu
        C_tied += Cc * Dc.shape[1]
    C_tied /= DTR.shape[1]
    return {c: (means[c], C_tied) for c in sorted(means)}

def naive_params(DTR, LTR):
    out = {}
    for c in sorted(set(LTR)):
        mu, C = compute_mu_C(DTR[:, LTR==c])
        out[c] = (mu, C * np.eye(DTR.shape[0]))
    return out

def compute_llr(D, params):

    """
    Computing Log-Likelihood Ratio (llr)

    pdf_GAU_ND( Class 1) / pdf_GAU_ND( Class 0 )  ---log--->  logpdf_GAU_ND - log-df_GAU_ND
    """
    mu1, C1 = params[1]
    mu0, C0 = params[0]
    return logpdf_GAU_ND(D, mu1, C1) - logpdf_GAU_ND(D, mu0, C0)

params_mvg   = mvg_params(DTR, LTR)
params_tied  = tied_params(DTR, LTR)
params_naive = naive_params(DTR, LTR)

llr_mvg   = compute_llr(DVAL, params_mvg)
llr_tied  = compute_llr(DVAL, params_tied)
llr_naive = compute_llr(DVAL, params_naive)

"""
llr: Log-Likelihood Ratio

llr  >   threshold --> class 1
llr  <   threshold --> class 0
"""
print('LLRs computed for all 3 models')

## Five applications → effective priors

Any application $(\pi_1, C_{fn}, C_{fp})$ is equivalent to an application $(\tilde{\pi}, 1, 1)$ where:

$$\tilde{\pi} = \frac{\pi_1 C_{fn}}{\pi_1 C_{fn} + (1 - \pi_1) C_{fp}}$$

This effective prior absorbs the costs into a single number. 

**Strong security** (high $C_{fp}$) maps to a **lower** effective prior for class 1 — the system is biased to reject, as if genuine users were rare.

**Ease of use** (high $C_{fn}$) maps to a **higher** effective prior — biased to accept.

In [ ]:
apps = [
    (0.5, 1.0, 1.0, 'Uniform prior & costs'),
    (0.9, 1.0, 1.0, 'High genuine prior'),
    (0.1, 1.0, 1.0, 'High fake prior'),
    (0.5, 1.0, 9.0, 'Strong security (high Cfp)'),
    (0.5, 9.0, 1.0, 'Ease of use (high Cfn)'),
]

print(f'{'Application':>20} {'Effective Prior':>25}')
print('-' * 48)
for pi1, Cfn, Cfp, label in apps:
    eff_prior = (pi1 * Cfn) / (pi1 * Cfn + (1 - pi1) * Cfp)
    print(f'({pi1}, {Cfn}, {Cfp})  {label:<22}  eff_prior = {eff_prior:.4f}')

As we can see the **5** different applications, become **3** different effective priors, which are `0.1`, `0.9` and `0.5`

## actDCF and minDCF for three applications

We evaluate three applications expressed as effective priors: $\tilde{\pi} = 0.1$, $0.5$, $0.9$.

- **actDCF** (actual DCF): DCF when using threshold - reflects both discriminative ability and calibration quality.

$$t = -\log\frac{\tilde{\pi}}{1-\tilde{\pi}}$$



- **minDCF** (minimum DCF): best possible DCF over all thresholds - reflects only discriminative ability, ignoring calibration.


<br>

-  **calibration loss** = actDCF - minDCF. A well-calibrated model has a small gap.

In [ ]:
three_apps = [(0.1, 1.0, 1.0), (0.5, 1.0, 1.0), (0.9, 1.0, 1.0)]
models = {'MVG': llr_mvg, 'Tied Gaussian': llr_tied, 'Naive Bayes': llr_naive}

rows = []
for model_name, llr in models.items():
    row = {'Model': model_name}
    for pi1, Cfn, Cfp in three_apps:
        pred = compute_optimal_Bayes_binary_llr(llr, pi1, Cfn, Cfp)

        """
        - What it does:

            This function takes the hard predictions (pred) --> compute_optimal_Bayes_binary_llr(llr, pil, Cfn, Cfp)
            compares them to the real ground-truth labels (LVAL),
            counts up the False Positives and False Negatives,
            and weights them by your costs to spit out a single penalty score
        
        - What it means:
            This is your Reality Check.
            It represents the actual, real-world penalty your system would pay
            if you deployed it today and blindly trusted its theoretical math.
        """
        act  = compute_empirical_Bayes_risk_binary(pred, LVAL, pi1, Cfn, Cfp)

        """
        - What it does: 
        
            It takes the raw llr scores. It completely ignores the theoretical threshold.
            Instead, it sweeps through every single possible threshold,
            tests them all, and finds the absolute lowest possible cost
            it can achieve on your validation set.
        
        - What it means: 
        
            This is Best-Case Scenario.
            It represents the ultimate discriminative power of your model.
            It asks:
             
                "If I had magically known the absolute perfect threshold
                to use for this specific dataset, what is the lowest penalty
                I could have possibly achieved?"
        """

        mn   = compute_minDCF_binary_fast(llr, LVAL, pi1, Cfn, Cfp)
        row[f'π̃={pi1} actDCF'] = round(act, 3)
        row[f'π̃={pi1} minDCF'] = round(mn, 3)
    rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
display(df)

### Answer: model comparison

**Best model by minDCF:** MVG and Naive Bayes are close and both outperform Tied Gaussian across all three applications. MVG is slightly better at $\tilde{\pi}=0.5$ and $0.9$; Naive Bayes is very close and occasionally matches MVG.

**Consistency:** The ranking MVG ≈ Naive Bayes > Tied is consistent across all three applications.

**Calibration:** For $\tilde{\pi}=0.5$ (the central application), all three models show a small calibration gap (actDCF - minDCF < 0.01–0.02), meaning the scores are well-calibrated near the balanced operating point. For extreme priors ($\tilde{\pi}=0.1$ and $0.9$), the calibration gap increases — the theoretical threshold is no longer optimal because the model parameters were estimated under the training distribution. Tied Gaussian shows the largest calibration loss overall.

## Bayes error plots

The Bayes error plot shows actDCF and minDCF as a function of prior log-odds $\tilde{p} = \log\frac{\tilde{\pi}}{1-\tilde{\pi}}$ over the range $[-4, +4]$.

- The **minimum DCF curve** (dashed) shows how well the model discriminates at each operating point.
- The **actual DCF curve** (solid) shows what the model achieves with the theoretical threshold.
- The **gap** between the two curves is the calibration loss at each operating point.

A well-calibrated model has solid and dashed curves that track closely together across the entire range.

In [ ]:
effPriorLogOdds = np.linspace(-4, 4, 100)
effPriors = 1.0 / (1.0 + np.exp(-effPriorLogOdds))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_styles = {
    'MVG':         ('tab:blue',  'tab:cyan'),
    'Tied Gaussian': ('tab:red', 'tab:orange'),
    'Naive Bayes': ('tab:green', 'tab:olive'),
}
for i, (model_name, llr) in enumerate({'MVG': llr_mvg, 'Tied Gaussian': llr_tied, 'Naive Bayes': llr_naive}.items()):
    actDCFs, minDCFs = [], []
    for ep in effPriors:
        pred = compute_optimal_Bayes_binary_llr(llr, ep, 1.0, 1.0)
        actDCFs.append(compute_empirical_Bayes_risk_binary(pred, LVAL, ep, 1.0, 1.0))
        minDCFs.append(compute_minDCF_binary_fast(llr, LVAL, ep, 1.0, 1.0))
    c_act, c_min = model_styles[model_name]
    ax = axes[i]
    ax.plot(effPriorLogOdds, actDCFs, color=c_act, label='act DCF', linewidth=2)
    ax.plot(effPriorLogOdds, minDCFs, color=c_min, label='min DCF', linewidth=2, linestyle='--')
    ax.set_title(model_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('prior log-odds'); ax.set_ylabel('DCF')
    ax.set_xlim([-4, 4]); ax.set_ylim([0, 1.1])
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Combined plot for direct model comparison
fig2, ax2 = plt.subplots(figsize=(10, 6))
style = {'MVG': ('tab:blue','tab:cyan','--'), 'Tied Gaussian': ('tab:red','tab:orange','-.'), 'Naive Bayes': ('tab:green','tab:olive',':')}
for model_name, llr in {'MVG': llr_mvg, 'Tied Gaussian': llr_tied, 'Naive Bayes': llr_naive}.items():
    actDCFs, minDCFs = [], []
    for ep in effPriors:
        pred = compute_optimal_Bayes_binary_llr(llr, ep, 1.0, 1.0)
        actDCFs.append(compute_empirical_Bayes_risk_binary(pred, LVAL, ep, 1.0, 1.0))
        minDCFs.append(compute_minDCF_binary_fast(llr, LVAL, ep, 1.0, 1.0))
    c_act, c_min, ls = style[model_name]
    ax2.plot(effPriorLogOdds, actDCFs, color=c_act, label=f'act DCF ({model_name})', linewidth=2)
    ax2.plot(effPriorLogOdds, minDCFs, color=c_min, label=f'min DCF ({model_name})', linewidth=2, linestyle=ls)
ax2.set_xlabel('prior log-odds'); ax2.set_ylabel('DCF value')
ax2.set_title('Bayes Error Plot — All Models')
ax2.set_xlim([-4, 4]); ax2.set_ylim([0, 1.1])
ax2.legend(loc='upper center', ncol=2, fontsize=9); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Final discussion

**Model rankings (minDCF):** MVG and Naive Bayes are consistently better than Tied Gaussian across the entire range of applications. This is consistent with the Lab 5 findings: the Tied assumption forces a shared covariance matrix, losing discriminative information from the different class-specific covariance structures (especially for features 5 and 6).

**Calibration:** All three models are reasonably well-calibrated near $\tilde{p} = 0$ (balanced application). As we move toward the extremes ($\tilde{p} \to \pm 4$), the actual DCF diverges from the minimum DCF — the models are trained under a balanced distribution but tested at extreme priors. The Tied Gaussian shows the largest calibration gap; MVG and Naive Bayes are better calibrated overall.

**Conclusion:** For this dataset, MVG and Naive Bayes are the best performing models and produce better-calibrated scores. Tied Gaussian is consistently inferior in both discriminative ability and calibration.